In [ ]:
"""
Program Name: dynamic_maze_solver.py
Description: Proof of concept for a dynamic maze solver using flood-fill algorithm.
             Assumes the solver does not have prior knowledge of the maze layout.
Programmer(s): Naran Bat
Date Made: 2/16/2025
Date(s) Revised:3/30/2025 - Added method to get data from lidar
                3/30/2025 - Added method to calculate goal position
                4/21/2025 - Rewrote method to recalculate path using BFS,  
                            replaced lidar with ultrasonic distance sensor
                4/23/2025 - Rewrote move_robot method to use movement functions
                4/24/2025 - Updated move_robot to use Moverment.py functions
Preconditions: 
Postconditions: 
Errors/Exceptions:
Side Effects:
Invariants: 
Known Faults:
"""

print("Hello")
import time
from collections import deque
import numpy as np
import nbformat

%run "../peripherals/Ultrasonic code.ipynb"

print("imported Ultrasonic")
with open("../movement/Helper_functions_movement.ipynb") as f:
    nb = nbformat.read(f, as_version=4)

# Run all code cells
for cell in nb.cells:
    if cell.cell_type == 'code':
        exec(cell.source)

print("imported movement")
# from ../peripherals/ultrasonic.ipynb import measure_distance  # Import the ultrasonic function
# from ../movement/movement.ipynb import init_wheels, forward, turn_left, turn_right, reverse, stop



DIRECTIONS = [(0, 1), (1, 0), (0, -1), (-1, 0)]  # Right, Down, Left, Up

class DynamicMazeSolver:
    def __init__(self, start, goal_angle, goal_distance):
        self.robot_position = np.array(start, dtype=float) # Start position
        self.goal_position = self.calculate_goal_position(goal_angle, goal_distance) # Goal position
        self.known_walls = set() # Set to store known walls
        self.path = [] # Path to goal
        self.explored = set()   

    def calculate_goal_position(self, angle, distance):
        """Converts polar coordinates (angle, distance) into Cartesian coordinates."""
        goal_x = self.robot_position[0] + distance * np.cos(angle) 
        goal_y = self.robot_position[1] + distance * np.sin(angle)
        return np.array([goal_x, goal_y])

    def heuristic(self, position):
        """Manhattan distance from position to goal."""
        return abs(position[0] - self.goal_position[0]) + abs(position[1] - self.goal_position[1])
    
    def update_walls(self):
        """Uses Ultrasonic readings to detect nearby walls and update known_walls set."""
        # Forward
        #distance = measure_distance()
        distance = 1
        if distance < 0.3:
            forward_pos = tuple(np.round(self.robot_position + np.array([0, 1])).astype(int))
            self.known_walls.add(forward_pos)
    
        # Check right
        turnRight(3)
        time.sleep(0.2)
        #distance = measure_distance()
        if distance < 0.3:
            right_pos = tuple(np.round(self.robot_position + np.array([1, 0])).astype(int))
            self.known_walls.add(right_pos)
        turnLeft(3) 
        time.sleep(0.2)
    
        # Check left
        turnLeft(3)
        time.sleep(0.2)
        #distance = measure_distance()
        if distance < 0.3:
            left_pos = tuple(np.round(self.robot_position + np.array([-1, 0])).astype(int))
            self.known_walls.add(left_pos)
        turnRight(3) 
        time.sleep(0.2)
    
    def bfs_recalculate_path(self):
        """Recalculates the best path using BFS based on discovered walls."""
        start = tuple(np.round(self.robot_position).astype(int))
        goal = tuple(np.round(self.goal_position).astype(int))
        queue = deque([(start, [])])
        visited = set()

        while queue:
            current_pos, path = queue.popleft()
            if np.linalg.norm(np.array(current_pos) - self.goal_position) < 0.1:
                return path  # Return shortest path found

            if current_pos in visited:
                continue
            visited.add(current_pos)

            for dr, dc in DIRECTIONS:
                next_pos = (current_pos[0] + dr, current_pos[1] + dc)
                if next_pos not in self.known_walls and next_pos not in visited:
                    queue.append((next_pos, path + [next_pos]))

        return []  # No path found



    def move_robot(self):
        """Moves the robot optimistically toward the goal."""
        print("inside mv")
        while np.linalg.norm(self.robot_position - self.goal_position) > 0.5:
            print("\nCurrent Robot Position:", np.round(self.robot_position, 2))
            self.update_walls()
            time.sleep(0.5)

            if not self.path:
                print("Recalculating path...")
                self.path = self.bfs_recalculate_path()
                if not self.path:
                    print("No available path to goal.")
                    stop()
                    return

            next_step = self.path.pop(0)
            direction = tuple(np.round(next_step - self.robot_position).astype(int))

            # Call motor control based on direction
            if direction == (0, 1): # Move forward
                moveForward(1)
            elif direction == (1, 0): # Move right
                turnRight(3)
                moveForward(1)
            elif direction == (0, -1): # Move left
                turnLeft(3)
                moveForward(1)
            elif direction == (-1, 0): # Move backward
                moveBackward(1)
            else:
                print("Invalid direction, stopping motors.")
                stop()
                return
        
            self.robot_position = np.array(next_step, dtype=float) # Update robot position
            self.explored.add(tuple(next_step)) # Add to explored set
            stop()
            time.sleep(0.5)

        print("Robot reached the goal!")
        stop()

# Initialize robot's start position and goal in polar coordinates
start_position = (0, 0)
goal_angle = 2  # radians
goal_distance = 10  # Meters

print("Hello")
solver = DynamicMazeSolver(start_position, goal_angle, goal_distance)
print("bye:")
solver.move_robot()
